# Lesson 29: Causal Inference

*ML Academy — companion notebook*

Prediction versus causation: confounding, randomised experiments, regression adjustment, propensity scores, difference-in-differences and uplift modelling.

Run each cell in order with **Shift+Enter**. The explanations, maths and worked examples are in `lesson-29.html`; this notebook contains every code example from the lesson so you can run and modify it.

Requirements: `pip install numpy pandas scikit-learn matplotlib`

## 2. Potential outcomes

In [1]:
import numpy as np
import pandas as pd
rng = np.random.default_rng(1)
n = 5000
loyalty = rng.normal(0, 1, n)                                   # the confounder
discount = rng.random(n) < 1 / (1 + np.exp(-1.5 * loyalty))     # loyal customers get more discounts
spend = 100 + 25 * loyalty + 10 * discount + rng.normal(0, 15, n)   # true effect of a discount: +10
df = pd.DataFrame({"loyalty": loyalty, "discount": discount.astype(int), "spend": spend})
naive = df[df.discount == 1].spend.mean() - df[df.discount == 0].spend.mean()
print("share given a discount:", round(df.discount.mean(), 3))
print("naive difference in spend:", round(naive, 1), "(the true effect is 10)")

share given a discount: 0.489
naive difference in spend: 35.7 (the true effect is 10)


## 4. The gold standard: randomised experiments

In [2]:
rand = rng.random(n) < 0.5                                       # a coin decides who gets the discount
spend_rct = 100 + 25 * loyalty + 10 * rand + rng.normal(0, 15, n)
diff = spend_rct[rand].mean() - spend_rct[~rand].mean()
se = np.sqrt(spend_rct[rand].var(ddof=1) / rand.sum() + spend_rct[~rand].var(ddof=1) / (~rand).sum())
print("randomised experiment:", round(diff, 1), "  95% interval", round(diff - 1.96 * se, 1), "to", round(diff + 1.96 * se, 1))

randomised experiment: 10.9   95% interval 9.3 to 12.6


## 5. Adjusting for confounders

In [3]:
from sklearn.linear_model import LinearRegression, LogisticRegression
adj = LinearRegression().fit(df[["discount", "loyalty"]], df.spend)
print("regression adjusting for loyalty:", round(adj.coef_[0], 1))
ps = LogisticRegression().fit(df[["loyalty"]], df.discount).predict_proba(df[["loyalty"]])[:, 1]
t, y = df.discount.to_numpy(), df.spend.to_numpy()
ipw = np.mean(t * y / ps) - np.mean((1 - t) * y / (1 - ps))      # inverse propensity weighting
print("propensity scores range from", round(ps.min(), 3), "to", round(ps.max(), 3))
print("inverse propensity weighting:", round(ipw, 1))
psc = np.clip(ps, 0.05, 0.95)                                    # trim extreme weights
w1, w0 = t / psc, (1 - t) / (1 - psc)
ipw_trim = np.sum(w1 * y) / w1.sum() - np.sum(w0 * y) / w0.sum()  # normalised (Hajek) weights
print("trimmed, normalised weighting:", round(ipw_trim, 1))

regression adjusting for loyalty: 10.0
propensity scores range from 0.006 to 0.996
inverse propensity weighting: 11.4
trimmed, normalised weighting: 10.0


## 6. Difference-in-differences

In [4]:
# Two cities; city B starts a loyalty app in month 7. Both share a seasonal trend.
months = np.arange(1, 13)
trend = 50 + 2 * months + 5 * np.sin(months / 2)
city_a = trend + rng.normal(0, 1, 12)
city_b = trend + 20 + 8 * (months >= 7) + rng.normal(0, 1, 12)    # true effect: +8
before, after = months < 7, months >= 7
naive_b = city_b[after].mean() - city_b[before].mean()
did = naive_b - (city_a[after].mean() - city_a[before].mean())
print("city B, after minus before:", round(naive_b, 1))
print("difference-in-differences:", round(did, 1), "(the true effect is 8)")

city B, after minus before: 13.4
difference-in-differences: 7.5 (the true effect is 8)


## 7. Who benefits? Uplift modelling

In [5]:
from sklearn.ensemble import GradientBoostingRegressor
# A coupon only helps price-sensitive customers; a randomised test is used to learn who.
m = 8000
sens = rng.random(m); age = rng.normal(40, 12, m)
coupon = rng.random(m) < 0.5
buy = (rng.random(m) < 0.10 + 0.20 * sens * coupon + 0.002 * (age - 40)).astype(int)
X = np.c_[sens, age]
tr, te = np.arange(m) < 6000, np.arange(m) >= 6000
m1 = GradientBoostingRegressor(max_depth=2, random_state=0).fit(X[tr & coupon], buy[tr & coupon])
m0 = GradientBoostingRegressor(max_depth=2, random_state=0).fit(X[tr & ~coupon], buy[tr & ~coupon])
uplift = m1.predict(X[te]) - m0.predict(X[te])                    # the "two-model" (T-learner) estimate
top = uplift >= np.quantile(uplift, 0.7)
def lift(mask):
    c, b = coupon[te][mask], buy[te][mask]
    return b[c].mean() - b[~c].mean()
print("coupon effect, top 30% by predicted uplift:", round(lift(top), 3))
print("coupon effect, the other 70%:", round(lift(~top), 3))
print("correlation of predicted uplift with sensitivity:", round(np.corrcoef(uplift, sens[te])[0, 1], 2))

coupon effect, top 30% by predicted uplift: 0.179
coupon effect, the other 70%: 0.09
correlation of predicted uplift with sensitivity: 0.64


## 9. In Python: a reusable effect estimator

In [6]:
import numpy as np
from sklearn.linear_model import LinearRegression, LogisticRegression

def estimate_effect(X, t, y):
    naive = y[t == 1].mean() - y[t == 0].mean()
    reg = LinearRegression().fit(np.c_[t, X], y).coef_[0]
    ps = np.clip(LogisticRegression().fit(X, t).predict_proba(X)[:, 1], 0.05, 0.95)
    w1, w0 = t / ps, (1 - t) / (1 - ps)
    ipw = (w1 * y).sum() / w1.sum() - (w0 * y).sum() / w0.sum()
    return {"naive": round(float(naive), 2), "regression": round(float(reg), 2), "weighting": round(float(ipw), 2)}

rng = np.random.default_rng(2)
X = rng.normal(0, 1, (3000, 2))                                 # two confounders
t = (rng.random(3000) < 1 / (1 + np.exp(-(X[:, 0] + X[:, 1])))).astype(int)
y = 5 + 3 * X[:, 0] + 2 * X[:, 1] + 4 * t + rng.normal(0, 2, 3000)   # true effect: 4
print(estimate_effect(X, t, y))

{'naive': 7.54, 'regression': 3.94, 'weighting': 4.0}


## 12. Going deeper: bad controls and regression discontinuity

In [7]:
import numpy as np
from sklearn.linear_model import LinearRegression
rng = np.random.default_rng(0); n = 20000
discount = rng.integers(0, 2, n)                               # randomised
visits = 2 + 3 * discount + rng.normal(0, 1, n)                 # the discount brings people back (a mediator)
spend = 50 + 4 * visits + rng.normal(0, 5, n)                   # all of the effect goes through visits: 4 × 3 = 12
total = LinearRegression().fit(np.c_[discount], spend).coef_[0]
blocked = LinearRegression().fit(np.c_[discount, visits], spend).coef_[0]
print("effect of the discount, not adjusting:", round(total, 2), "  'adjusting' for visits:", round(blocked, 2))

effect of the discount, not adjusting: 11.96   'adjusting' for visits: -0.08


In [8]:
# Regression discontinuity: customers with a score of 600 or more get a credit limit increase.
score = rng.uniform(450, 750, n)
treated = (score >= 600).astype(int)
spend2 = 20 + 0.1 * score + 15 * treated + rng.normal(0, 10, n)   # true jump: 15
naive2 = spend2[treated == 1].mean() - spend2[treated == 0].mean()
band = np.abs(score - 600) < 25                                     # only customers near the threshold
Xb = np.c_[treated[band], score[band] - 600, treated[band] * (score[band] - 600)]
rdd = LinearRegression().fit(Xb, spend2[band]).coef_[0]
print("naive difference:", round(naive2, 1), "  regression discontinuity (±25 points):", round(rdd, 1))

naive difference: 29.8   regression discontinuity (±25 points): 14.7


## Running example: Chennai Homes

The same 12-house dataset used in every lesson (`data/houses.csv` in the course folder).

In [9]:
import pandas as pd
import numpy as np
df = pd.read_csv("../data/houses.csv")
from sklearn.linear_model import LinearRegression
df["metro"] = df.description.str.contains("metro").astype(int)
naive = df[df.metro == 1].sold_fast.mean() - df[df.metro == 0].sold_fast.mean()
X = pd.get_dummies(df[["metro", "area"]], columns=["area"], drop_first=True, dtype=float)
adjusted = LinearRegression().fit(X, df.sold_fast).coef_[0]
print("homes near a metro:", df.metro.sum(), "of", len(df))
print("naive difference in sold-fast rate:", round(naive, 2), "  adjusted for area:", round(adjusted, 2))

homes near a metro: 3 of 12
naive difference in sold-fast rate: 0.67   adjusted for area: 1.0


## Coding challenges

Each cell below has **one missing line**, marked `# ← your code here` (or `# ← replace`). Write the line, run the cell, and the check at the bottom tells you whether you got it right. Stuck? A wrong answer prints a hint, and the solutions are at the very end of this notebook.

In [10]:
# Checker used by the challenges below — run this cell first.
import numpy as np
def check(got, want, hint='', atol=1e-6):
    try:
        ok = got is not None and np.shape(got) == np.shape(want) and np.allclose(np.asarray(got, dtype=float), np.asarray(want, dtype=float), rtol=1e-4, atol=atol)
    except Exception:
        ok = False
    print('✅ Correct!' if ok else '❌ Not yet. Hint: ' + hint)


### Challenge 1: Difference-in-differences

Compute the DiD estimate from the four group averages.

In [11]:
treated_before, treated_after = 50.0, 70.0
control_before, control_after = 48.0, 58.0
did = treated_after - treated_before  # ← subtract the control group's change
check(did, 10.0, hint="Subtract the control group's change from the treated group's change.")

❌ Not yet. Hint: Subtract the control group's change from the treated group's change.


### Challenge 2: Inverse propensity weights

Compute the weight for each person: 1/p if treated, 1/(1 − p) if not.

In [12]:
import numpy as np
t = np.array([1, 0, 1, 0])
p = np.array([0.8, 0.8, 0.25, 0.5])
w = 1 / p  # ← untreated people need 1 / (1 - p)
result = w.tolist()
check(result, [1.25, 5.000000000000001, 4.0, 2.0], hint='Use np.where on the treatment flag.')

❌ Not yet. Hint: Use np.where on the treatment flag.


## Your turn

Try the **practice exercises** at the end of the lesson page. Many of them ask you to change one of the cells above and re-run it.

## Challenge solutions

*Spoilers — try the challenges first!*

**1. Difference-in-differences** — the missing line:

```python
did = (treated_after - treated_before) - (control_after - control_before)
```

**2. Inverse propensity weights** — the missing line:

```python
w = np.where(t == 1, 1 / p, 1 / (1 - p))
```